In [1]:
!pip install transformers datasets -q

In [2]:
import time
from datasets import load_dataset
from transformers import pipeline

print('Loading summarization pipeline...')
summarizer = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6')
print('Pipeline ready.')

## Load a Small Sample

In [3]:
dataset = load_dataset('cnn_dailymail', '3.0.0', split='test[:5]')
print('Articles loaded:', len(dataset))
print('Example title:', dataset[0]['article'][:120].replace('\n', ' ') + '...')

## Generate Summaries

In [4]:
def summarize_batch(texts, max_len=130, min_len=30):
    outputs = []
    start = time.time()
    for txt in texts:
        summary = summarizer(txt, max_length=max_len, min_length=min_len, do_sample=False)[0]['summary_text']
        outputs.append(summary)
    elapsed = time.time() - start
    return outputs, elapsed

articles = dataset['article']
summaries, elapsed = summarize_batch(articles)

for i, (art, summ) in enumerate(zip(articles, summaries), 1):
    print(f"\n=== Sample {i} ===")
    print('Article snippet:', art[:200].replace('\n', ' ') + '...')
    print('Summary:', summ)

print(f"\nTotal time for {len(articles)} summaries: {elapsed:.2f}s")

## Quick Quality Check

In [5]:
def compression_ratio(src, tgt):
    return len(tgt.split()) / max(1, len(src.split()))

ratios = [compression_ratio(a, s) for a, s in zip(articles, summaries)]
print('Compression ratios:', [f"{r:.2f}" for r in ratios])
print('Average ratio:', sum(ratios) / len(ratios))

## Test on Custom Text

In [6]:
custom_article = '''
Transformers have become the dominant architecture for natural language processing tasks.
By relying on self-attention, they capture long-range dependencies efficiently.
This has led to breakthroughs in translation, summarization, and question answering.
'''
custom_summary = summarizer(custom_article, max_length=80, min_length=25, do_sample=False)[0]['summary_text']
print('Custom summary:', custom_summary)